=====================================================================
기본_모형_AR_일반LAD.py

목적: "기본 모형 AR" 을, bounded LAD(학습 중 [0,1] 강제 - 논문에 없는
      operational_corrected 만의 조건, D:\03_JiWon\APEN\docs\
      02_RESULTS_AND_LIMITATIONS.md §6.4 에서 스스로 "논문에 없다"고
      밝힌 부분) 대신, 논문 Eq.(5) 에 실제로 적힌 절대오차합(LAD)만
      최소화하는 "일반 LAD" 방식으로 다시 구한다.

      즉 학습 중에는 예측값이 [0,1] 을 벗어나도 그냥 두고(제약 없음),
      테스트 예측이 끝난 뒤에만 물리적으로 말이 되게 [0,1] 로 잘라낸다
      (사후 clip). 이게 sklearn QuantileRegressor 와 같은 방식이다.

      기본_모형_AR.py(bounded LAD, 결과표 공식 버전)와 비교하기 위한
      파일이며, 5번 구간(회귀 푸는 부분)만 다르고 나머지는 전부 동일.

코딩 스타일: class, def(함수) 를 전혀 쓰지 않는다. 위에서 아래로
            순서대로 실행되는 코드만 쓴다(naive 스타일). 거의 모든
            줄에 그 줄이 뭘 하는지 주석을 단다.

데이터: merged_for_simulation_z01.csv (Zone1, EST->UTC 보정,
        dssrd/dtsr 차분, Sydney 현지시간 열 포함)
구간(블록9): 학습 2012-11-28~2013-09-23(300일),
            테스트 2013-09-24~2014-01-01(100일)
=====================================================================

In [2]:
import os                                    # 파일 경로를 다루는 표준 라이브러리
import numpy as np                           # 숫자 배열(행렬) 계산 라이브러리
import pandas as pd                          # 표(csv) 데이터를 다루는 라이브러리
from scipy import sparse                     # 희소행렬(linprog 제약식용) 라이브러리
from scipy.optimize import linprog           # 선형계획법(LP) 솔버 - LAD 회귀를 푸는 데 씀

In [4]:
# =====================================================================
# 0. 설정값 (여기 숫자만 바꾸면 동작이 바뀜)
# =====================================================================
#BASE_DIR = os.path.dirname(os.path.abspath(__file__))           # 이 파이썬 파일이 있는 폴더 경로
#MERGED_FILE = os.path.join(BASE_DIR, "merged_for_simulation_z01.csv")  # 읽어올 병합 데이터 파일 경로

BASE_DIR = os.getcwd()                                            # 현재 작업 디렉토리(Current Working Directory) 경로
MERGED_FILE = os.path.join(BASE_DIR, "merged_for_simulation_z01.csv")  # 읽어올 병합 데이터 파일 경로

In [5]:
HOURS_PER_DAY = 12               # 하루 낮 시간대 개수 (Sydney 현지시간 9시~20시)
LOCAL_HOUR_START = 9             # 낮 시간대 시작 시(local_hour 기준)
LOCAL_HOUR_END = 21              # 낮 시간대 끝(이 값 미만까지, 즉 9~20시)

In [6]:
TRAIN_START = pd.Timestamp("2012-11-28")   # 학습 시작일
TRAIN_END = pd.Timestamp("2013-09-23")     # 학습 마지막일 (300일째)
TEST_START = pd.Timestamp("2013-09-24")    # 테스트 시작일
TEST_END = pd.Timestamp("2014-01-01")      # 테스트 마지막일 (100일째)
HISTORY_DATE = pd.Timestamp("2012-11-27")  # 학습 첫날의 "직전 하루" (AR 입력 lag용, target 아님)

In [8]:
CAPACITY_MW = 30.0                # 태양광 패널 설비 최대 용량 (논문 가정)
DURATION_HOURS = 1.0              # 한 시간대의 길이(시간)
PENALTY_RATE = 0.5                # 약정 부족(shortage) 시 벌금비용률 (일간전 가격의 50%)

In [9]:
PAPER_NRMSE = 34.76                # 논문 Table 3, 기본 모형 AR 의 nRMSE(%) - 비교용
PAPER_GAP = 15.04                  # 논문 Table 3, 기본 모형 AR 의 optimality gap(%) - 비교용

In [10]:
BOUNDED_LAD_NRMSE = 38.783485      # 기본_모형_AR.py(bounded LAD) 결과 - 비교용
BOUNDED_LAD_GAP = 3.696468         # 기본_모형_AR.py(bounded LAD) 결과 - 비교용

In [11]:
# =====================================================================
# 1. 데이터 읽기 + Sydney 현지시간 낮 시간대만 남기기
# =====================================================================
raw_table = pd.read_csv(MERGED_FILE)                       # csv 파일 전체를 한 번에 읽어옴
raw_table["local_date"] = pd.to_datetime(raw_table["local_date"])   # local_date 열을 날짜 타입으로 변환

In [12]:
is_daylight = (raw_table["local_hour"] >= LOCAL_HOUR_START) & (raw_table["local_hour"] < LOCAL_HOUR_END)
# ↑ local_hour 가 9시 이상, 21시 미만(=9~20시)인 행만 True 인 판단 열을 만듦

In [53]:
is_daylight

0         True
1         True
2         True
3         True
4         True
         ...  
13094    False
13095    False
13096    False
13097    False
13098     True
Name: local_hour, Length: 13099, dtype: bool

In [15]:
daylight_table = raw_table[is_daylight].copy()              # 낮 시간대 행만 골라서 새 표로 복사
daylight_table["hour_idx"] = daylight_table["local_hour"] - LOCAL_HOUR_START
# ↑ local_hour(9~20) 를 0~11 로 다시 번호 매김 (hour_idx)

In [14]:
daylight_table

,timestamp,solar_power,da_price,rt_price,tclw,tciw,sp,r,tcc,u10,...,ssrd,strd,tsr,tp,dssrd,dtsr,local_timestamp,local_date,local_hour,hour_idx
0,2012-11-01 05:00:00,0.195064,22.05,21.47,0.005581,0.000576,93677.0000,47.458649,0.067795,8.756279,...,15500361.0,5989990.0,17267728.0,0.0,2865314.0,3118999.0,2012-11-01 16:00:00,2012-11-01,16,7
1,2012-11-01 06:00:00,0.219551,21.68,20.58,0.001041,0.000000,93725.2500,51.379166,0.005356,7.914716,...,17723120.0,6990777.0,19665664.0,0.0,2222759.0,2397936.0,2012-11-01 17:00:00,2012-11-01,17,8
2,2012-11-01 07:00:00,0.061923,21.61,27.46,0.000484,0.000000,93768.1875,37.110474,0.005051,6.600511,...,19159296.0,7986873.0,21279120.0,0.0,1436176.0,1613456.0,2012-11-01 18:00:00,2012-11-01,18,9
3,2012-11-01 08:00:00,0.039679,22.08,20.77,0.000271,0.000000,93862.5625,45.098145,0.000000,5.325268,...,19748976.0,8975498.0,22026896.0,0.0,589680.0,747776.0,2012-11-01 19:00:00,2012-11-01,19,10
4,2012-11-01 09:00:00,0.004808,24.09,25.27,0.000084,0.000000,93959.1250,49.032501,0.000000,3.397720,...,19788400.0,9953138.0,22103168.0,0.0,39424.0,76272.0,2012-11-01 20:00:00,2012-11-01,20,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13082,2014-04-30 07:00:00,0.035833,23.86,27.29,0.006269,0.000203,94439.5000,45.626053,0.248352,1.686311,...,10813564.0,7540547.0,12302309.0,0.0,345005.0,480075.0,2014-04-30 17:00:00,2014-04-30,17,8
13083,2014-04-30 08:00:00,0.002436,24.16,27.04,0.008289,0.000089,94512.5625,50.893799,0.316742,1.817962,...,10826395.0,8565945.0,12328406.0,0.0,12831.0,26097.0,2014-04-30 18:00:00,2014-04-30,18,9
13084,2014-04-30 09:00:00,0.000000,26.75,28.29,0.005760,0.000000,94602.7500,57.247131,0.239944,1.659210,...,10826395.0,9562098.0,12328406.0,0.0,0.0,0.0,2014-04-30 19:00:00,2014-04-30,19,10
13085,2014-04-30 10:00:00,0.000000,37.73,34.22,0.000748,0.000000,94687.5625,61.398071,0.070007,0.951973,...,10826395.0,10551939.0,12328406.0,0.0,0.0,0.0,2014-04-30 20:00:00,2014-04-30,20,11


In [22]:
# =====================================================================
# 2. 이력(history) / 학습(train) / 테스트(test) 구간으로 자르기
# =====================================================================
is_history_date = daylight_table["local_date"] == HISTORY_DATE     # 이력 날짜(2012-11-27)인지 판단
history_rows = daylight_table[is_history_date].copy()               # 이력 날짜 행만 골라냄
history_rows = history_rows.sort_values("hour_idx")                  # 시간대(0~11) 순서로 정렬

In [23]:
#is_history_date
history_rows

,timestamp,solar_power,da_price,rt_price,tclw,tciw,sp,r,tcc,u10,...,ssrd,strd,tsr,tp,dssrd,dtsr,local_timestamp,local_date,local_hour,hour_idx
617,2012-11-26 22:00:00,0.083205,58.75,56.56,0.013840,0.035153,94541.8750,60.036072,0.644832,1.498468,...,20058528.0,28850192.0,24426752.0,0.000000,1532288.0,1910640.0,2012-11-27 09:00:00,2012-11-27,9,0
618,2012-11-26 23:00:00,0.139167,57.46,65.52,0.003014,0.007801,94550.7500,57.443527,0.316742,1.413047,...,21966880.0,30167888.0,26824640.0,0.000000,1908352.0,2397888.0,2012-11-27 10:00:00,2012-11-27,10,1
619,2012-11-27 00:00:00,0.196346,47.36,38.27,0.001026,0.000032,94509.5625,48.778671,0.045137,1.991235,...,24919760.0,31507936.0,30162128.0,0.000000,2952880.0,3337488.0,2012-11-27 11:00:00,2012-11-27,11,2
620,2012-11-27 01:00:00,0.344872,43.78,36.81,0.008847,0.016509,94528.3750,53.669617,0.286709,0.372191,...,2218507.0,1348963.0,2912659.0,0.000000,2218507.0,2912659.0,2012-11-27 12:00:00,2012-11-27,12,3
621,2012-11-27 02:00:00,0.545449,41.90,40.93,0.006820,0.029383,94448.3750,45.603180,0.225929,1.541364,...,5662094.0,2723651.0,6787185.0,0.000000,3443587.0,3874526.0,2012-11-27 13:00:00,2012-11-27,13,4
622,2012-11-27 03:00:00,0.388205,33.57,33.04,0.035134,0.098199,94414.1250,52.144806,0.728081,2.405942,...,9324023.0,4040254.0,10826937.0,0.000547,3661929.0,4039752.0,2012-11-27 14:00:00,2012-11-27,14,5
623,2012-11-27 04:00:00,0.158526,28.66,30.22,0.049652,0.251840,94475.8750,68.894043,0.792901,2.655906,...,11928426.0,5345407.0,14070259.0,0.002378,2604403.0,3243322.0,2012-11-27 15:00:00,2012-11-27,15,6
624,2012-11-27 05:00:00,0.123590,29.00,25.75,0.091917,0.187083,94456.4375,74.298569,0.964552,2.390555,...,13793237.0,6689769.0,16530443.0,0.004790,1864811.0,2460184.0,2012-11-27 16:00:00,2012-11-27,16,7
625,2012-11-27 06:00:00,0.019231,26.35,25.41,0.085347,0.330627,94466.3750,79.991974,0.880239,3.241761,...,14920313.0,8086680.0,18211504.0,0.008264,1127076.0,1681061.0,2012-11-27 17:00:00,2012-11-27,17,8
626,2012-11-27 07:00:00,0.012372,23.92,24.43,0.046108,0.133488,94454.0000,82.437119,0.832852,1.450317,...,15631916.0,9452046.0,19367360.0,0.009427,711603.0,1155856.0,2012-11-27 18:00:00,2012-11-27,18,9


In [24]:
is_train_date = (daylight_table["local_date"] >= TRAIN_START) & (daylight_table["local_date"] <= TRAIN_END)
train_rows = daylight_table[is_train_date].copy()                    # 학습 구간 행만 골라냄
train_rows = train_rows.sort_values(["local_date", "hour_idx"])       # 날짜, 시간대 순서로 정렬

In [25]:
is_test_date = (daylight_table["local_date"] >= TEST_START) & (daylight_table["local_date"] <= TEST_END)
test_rows = daylight_table[is_test_date].copy()                      # 테스트 구간 행만 골라냄
test_rows = test_rows.sort_values(["local_date", "hour_idx"])         # 날짜, 시간대 순서로 정렬

In [26]:
print("이력 날짜:", HISTORY_DATE.date(), "행 수:", len(history_rows))          # 이력 행 수 출력 (12여야 정상)
print("학습 구간:", TRAIN_START.date(), "~", TRAIN_END.date(), "행 수:", len(train_rows))  # 학습 행 수 출력 (3600 이어야 정상)
print("테스트 구간:", TEST_START.date(), "~", TEST_END.date(), "행 수:", len(test_rows))    # 테스트 행 수 출력 (1200 이어야 정상)

이력 날짜: 2012-11-27 행 수: 12
학습 구간: 2012-11-28 ~ 2013-09-23 행 수: 3600
테스트 구간: 2013-09-24 ~ 2014-01-01 행 수: 1200


=====================================================================
3. (날짜 x 12시간) 모양의 숫자 배열로 바꾸기
=====================================================================

In [27]:
# --- 3-1. 이력(history) 하루치 발전량 배열 (1, 12) ---
history_solar = np.zeros((1, HOURS_PER_DAY))          # 0으로 채운 빈 배열 준비 (1일 x 12시간)
row_counter = 0                                          # history_rows 를 순서대로 셀 카운터
for _, one_row in history_rows.iterrows():                # history_rows 를 한 줄씩 순서대로 확인
    hour_position = row_counter % HOURS_PER_DAY            # 이 행이 몇 번째 시간대인지 (0~11)
    history_solar[0, hour_position] = one_row["solar_power"]  # 해당 칸에 발전량 값을 채워 넣음
    row_counter = row_counter + 1                            # 카운터를 하나 증가시킴

In [29]:
hour_position
history_solar

array([[0.08320513, 0.13916667, 0.19634615, 0.34487179, 0.54544872,
        0.38820513, 0.15852564, 0.12358974, 0.01923077, 0.01237179,
        0.07      , 0.02352564]])

In [30]:
# --- 3-2. 학습(train) 300일치 발전량 배열 (300, 12) ---
train_dates_sorted = sorted(train_rows["local_date"].unique())  # 학습 구간의 날짜들을 오래된 순으로 정렬한 목록
n_train_days = len(train_dates_sorted)                            # 학습 날짜 수 (300 이어야 정상)
train_solar = np.zeros((n_train_days, HOURS_PER_DAY))              # 0으로 채운 빈 배열 준비 (300일 x 12시간)
row_counter = 0                                                      # train_rows 를 순서대로 셀 카운터
for _, one_row in train_rows.iterrows():                             # train_rows 를 한 줄씩 순서대로 확인
    day_position = row_counter // HOURS_PER_DAY                        # 이 행이 몇 번째 날짜인지 (0~299)
    hour_position = row_counter % HOURS_PER_DAY                        # 이 행이 몇 번째 시간대인지 (0~11)
    train_solar[day_position, hour_position] = one_row["solar_power"]   # 해당 칸에 발전량 값을 채워 넣음
    row_counter = row_counter + 1                                        # 카운터를 하나 증가시킴

In [31]:
# --- 3-3. 테스트(test) 100일치 발전량 / 가격 배열 (100, 12) ---
test_dates_sorted = sorted(test_rows["local_date"].unique())     # 테스트 구간의 날짜들을 오래된 순으로 정렬한 목록
n_test_days = len(test_dates_sorted)                               # 테스트 날짜 수 (100 이어야 정상)
test_solar = np.zeros((n_test_days, HOURS_PER_DAY))                 # 실제 발전량을 담을 빈 배열
test_da_price = np.zeros((n_test_days, HOURS_PER_DAY))              # 일간전(DA) 가격을 담을 빈 배열
test_rt_price = np.zeros((n_test_days, HOURS_PER_DAY))              # 실시간(RT) 가격을 담을 빈 배열
row_counter = 0                                                       # test_rows 를 순서대로 셀 카운터
for _, one_row in test_rows.iterrows():                               # test_rows 를 한 줄씩 순서대로 확인
    day_position = row_counter // HOURS_PER_DAY                          # 이 행이 몇 번째 날짜인지 (0~99)
    hour_position = row_counter % HOURS_PER_DAY                          # 이 행이 몇 번째 시간대인지 (0~11)
    test_solar[day_position, hour_position] = one_row["solar_power"]      # 실제 발전량 값을 채워 넣음
    test_da_price[day_position, hour_position] = one_row["da_price"]      # DA 가격 값을 채워 넣음
    test_rt_price[day_position, hour_position] = one_row["rt_price"]      # RT 가격 값을 채워 넣음
    row_counter = row_counter + 1                                          # 카운터를 하나 증가시킴

=====================================================================
4. AR 학습용 입력(X), 정답(y) 만들기 - 논문 Eq.(3): "직전 하루"의
   12시간을 입력으로 써서, 다음날 각 시간대를 예측한다.
=====================================================================

In [32]:
history_and_train_solar = np.vstack([history_solar, train_solar])   # 이력 하루 + 학습 300일을 위아래로 이어붙임 (301, 12)

In [37]:
history_solar

array([[0.08320513, 0.13916667, 0.19634615, 0.34487179, 0.54544872,
        0.38820513, 0.15852564, 0.12358974, 0.01923077, 0.01237179,
        0.07      , 0.02352564]])

In [33]:
n_ar_rows = n_train_days                                             # AR 학습 표본 개수 = 학습 날짜 수 (300개)
ar_intercept_column = np.ones((n_ar_rows, 1))                        # 절편(intercept)을 위한 1로만 채운 열 (300, 1)
ar_lag_features = np.zeros((n_ar_rows, HOURS_PER_DAY))                # "직전 하루" 12시간 값을 담을 빈 배열 (300, 12)
for day_index in range(n_ar_rows):                                     # 학습 날짜 0번째부터 299번째까지 순서대로
    previous_day_values = history_and_train_solar[day_index]            # 이 학습일의 "바로 전날" 12시간 값
    ar_lag_features[day_index] = previous_day_values[::-1]               # 시간을 거꾸로 뒤집어서 저장  ???

In [36]:
ar_intercept_column
previous_day_values
ar_lag_features

array([[2.35256410e-02, 7.00000000e-02, 1.23717950e-02, ...,
        1.96346154e-01, 1.39166667e-01, 8.32051280e-02],
       [1.32051280e-02, 4.02564100e-02, 1.08269231e-01, ...,
        2.17628205e-01, 1.28141026e-01, 2.17948718e-01],
       [1.46794870e-02, 3.77564100e-02, 7.58974360e-02, ...,
        7.47435897e-01, 6.64166667e-01, 3.26025641e-01],
       ...,
       [0.00000000e+00, 1.28205000e-04, 2.07051280e-02, ...,
        2.59935897e-01, 4.92820513e-01, 3.69487179e-01],
       [0.00000000e+00, 1.28205000e-04, 1.89102560e-02, ...,
        8.31025641e-01, 7.92115385e-01, 6.89230769e-01],
       [0.00000000e+00, 2.56410000e-04, 2.80128210e-02, ...,
        4.56987179e-01, 4.66089744e-01, 3.52115385e-01]], shape=(300, 12))

In [38]:
ar_design_matrix = np.hstack([ar_intercept_column, ar_lag_features])   # 절편 열 + lag 12개 열을 옆으로 이어붙임 (300, 13)

In [43]:
ar_design_matrix

array([[1.00000000e+00, 2.35256410e-02, 7.00000000e-02, ...,
        1.96346154e-01, 1.39166667e-01, 8.32051280e-02],
       [1.00000000e+00, 1.32051280e-02, 4.02564100e-02, ...,
        2.17628205e-01, 1.28141026e-01, 2.17948718e-01],
       [1.00000000e+00, 1.46794870e-02, 3.77564100e-02, ...,
        7.47435897e-01, 6.64166667e-01, 3.26025641e-01],
       ...,
       [1.00000000e+00, 0.00000000e+00, 1.28205000e-04, ...,
        2.59935897e-01, 4.92820513e-01, 3.69487179e-01],
       [1.00000000e+00, 0.00000000e+00, 1.28205000e-04, ...,
        8.31025641e-01, 7.92115385e-01, 6.89230769e-01],
       [1.00000000e+00, 0.00000000e+00, 2.56410000e-04, ...,
        4.56987179e-01, 4.66089744e-01, 3.52115385e-01]], shape=(300, 13))

=====================================================================
5. 시간대(h=0~11) 마다 따로 "일반 LAD" 회귀를 풀어서 계수를 구함
   - 일반 LAD = 절대오차합(논문 Eq.5)만 최소화, [0,1] 제약은 안 걸음
   - bounded LAD(기본_모형_AR.py)와 차이: 아래 constraint_block_3,
     constraint_block_4 (예측값을 [0,1]로 가두는 제약) 두 개가 없음
   - scipy.optimize.linprog 으로 직접 선형계획법을 풂
=====================================================================

In [52]:
n_features = ar_design_matrix.shape[1]                # 입력 변수 개수 (절편 포함 13개)
coefficients_by_hour = np.zeros((HOURS_PER_DAY, n_features))   # 시간대별 계수(13개)를 저장할 빈 배열 (12, 13)

In [45]:
coefficients_by_hour

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [46]:
for hour in range(HOURS_PER_DAY):                       # 시간대 0부터 11까지 순서대로 하나씩 처리

    y_this_hour = train_solar[:, hour]                    # 이 시간대의 정답값(실제 발전량) 300개

    X_sparse = sparse.csr_matrix(ar_design_matrix)          # 입력행렬을 희소행렬 형태로 변환 (linprog 입력용)
    identity_matrix = sparse.eye(n_ar_rows, format="csr")     # 300x300 단위행렬 (절대값 처리용 보조변수 계수)

    # ---- 절대오차 |y - X@beta| 를 u 라는 보조변수로 표현하기 위한 제약 2묶음만 씀 ----
    # (bounded 버전에 있던 "예측값을 [0,1]로 가두는" 제약 2묶음은 여기선 뺐음 -
    #  논문 Eq.5 에는 그런 제약이 없으므로, 있는 그대로의 LAD만 품)
    constraint_block_1 = sparse.hstack([X_sparse, -identity_matrix])   #  X@beta - u <= y
    constraint_block_2 = sparse.hstack([-X_sparse, -identity_matrix])  # -X@beta - u <= -y  (합치면 u >= |X@beta-y|)

    all_constraints = sparse.vstack([                        # 위 2묶음을 위아래로 합쳐 하나의 제약행렬로 만듦
        constraint_block_1, constraint_block_2
    ], format="csr")

    constraint_limits = np.concatenate([                     # 각 제약식의 우변(<=) 값들을 순서대로 이어붙임
        y_this_hour,                 # 첫 묶음의 우변 = y
        -y_this_hour,                # 두번째 묶음의 우변 = -y
    ])

    objective_coefficients = np.concatenate([                # 목적함수 계수: beta 에는 0, u 에는 1/300
        np.zeros(n_features),          # beta(13개) 는 목적함수에 직접 안 들어감
        np.ones(n_ar_rows) / n_ar_rows,   # u(300개)의 평균을 최소화 = 평균절대오차(LAD) 최소화
    ])

    variable_bounds = [(None, None)] * n_features + [(0.0, None)] * n_ar_rows
    # ↑ beta 는 부호 제한 없음(-무한대~+무한대), u 는 0 이상이어야 함(절대값이니까)

    lp_result = linprog(                                     # 선형계획법을 실제로 풂
        objective_coefficients,
        A_ub=all_constraints,
        b_ub=constraint_limits,
        bounds=variable_bounds,
        method="highs",                                        # highs 라는 빠른 LP 알고리즘 사용
    )

    coefficients_by_hour[hour] = lp_result.x[:n_features]     # 결과에서 beta(13개)만 뽑아 이 시간대의 계수로 저장

    print(f"  시간대 {hour} 회귀 완료 (성공 여부: {lp_result.success})")   # 진행상황 출력

  시간대 0 회귀 완료 (성공 여부: True)
  시간대 1 회귀 완료 (성공 여부: True)
  시간대 2 회귀 완료 (성공 여부: True)
  시간대 3 회귀 완료 (성공 여부: True)
  시간대 4 회귀 완료 (성공 여부: True)
  시간대 5 회귀 완료 (성공 여부: True)
  시간대 6 회귀 완료 (성공 여부: True)
  시간대 7 회귀 완료 (성공 여부: True)
  시간대 8 회귀 완료 (성공 여부: True)
  시간대 9 회귀 완료 (성공 여부: True)
  시간대 10 회귀 완료 (성공 여부: True)
  시간대 11 회귀 완료 (성공 여부: True)


In [51]:
type(coefficients_by_hour)
coefficients_by_hour


array([[ 5.99217833e-02,  1.13221520e+00, -1.81240584e-01,
         2.64277226e-01, -4.05247809e-01,  3.99389582e-01,
        -3.61789600e-03,  2.85997919e-02, -1.25720103e-01,
        -2.95156447e-03,  1.05604673e-01, -1.39125414e-01,
         7.96004976e-01],
       [ 2.32303892e-01,  1.29889995e+00, -7.88803421e-01,
        -1.45656061e+00, -4.09587758e-02,  4.34718613e-01,
        -9.46504841e-02,  4.24396158e-01, -1.27646506e-01,
        -2.34221789e-01,  2.54378442e-01,  1.04113998e-01,
         3.03422276e-01],
       [ 3.46342058e-01,  2.94097942e-02, -5.22871515e-01,
        -1.09556177e+00,  1.42651801e-01,  3.84344700e-01,
        -2.24441066e-01,  3.33397512e-01,  3.09568057e-02,
        -2.71756093e-01,  3.35264920e-01,  5.21948830e-02,
         2.08562385e-01],
       [ 3.86961712e-01,  6.70231972e-02,  9.25510047e-01,
        -1.10147452e+00, -3.24984327e-01,  7.99062207e-01,
        -3.84672442e-01,  4.73837410e-01, -7.81706584e-03,
        -7.20775714e-02,  1.32213530e

=====================================================================
6. 테스트 100일을 하루씩 순서대로 예측 (rolling one-day-ahead)
   - 예측값이 [0,1] 을 벗어나면 여기서만(사후에) 잘라냄
=====================================================================

In [50]:
test_forecast = np.zeros((n_test_days, HOURS_PER_DAY))     # 예측 결과를 담을 빈 배열 (100, 12)
previous_day_actual = train_solar[-1]                        # 테스트 첫날 예측에 쓸 "직전 하루" = 학습 마지막 날

In [ ]:
for day_index in range(n_test_days):                          # 테스트 0번째 날부터 99번째 날까지 순서대로

    feature_vector = np.concatenate([[1.0], previous_day_actual[::-1]])
    # ↑ [절편 1] + [직전 하루 12시간을 거꾸로 뒤집은 값]

    for hour in range(HOURS_PER_DAY):                            # 이 날의 시간대 0~11을 하나씩 예측
        raw_prediction = np.dot(coefficients_by_hour[hour], feature_vector)  # 계수와 입력을 곱해서 더함 (내적)
        clipped_prediction = min(max(raw_prediction, 0.0), 1.0)              # 예측값을 사후에 0~1 범위로 잘라냄
        test_forecast[day_index, hour] = clipped_prediction                    # 예측 결과 배열에 저장

    previous_day_actual = test_solar[day_index]                # 다음날 예측을 위해 "직전 하루"를 오늘의 실제값으로 갱신

=====================================================================
7. nRMSE 계산 (Eq. 11-12)
=====================================================================

In [ ]:
actual_flat = test_solar.flatten()          # (100,12) 실제값 표를 1200개짜리 한 줄로 펼침
predicted_flat = test_forecast.flatten()     # (100,12) 예측값 표도 1200개짜리 한 줄로 펼침

In [ ]:
sum_of_squared_error = 0.0                   # 제곱오차를 누적할 변수, 0에서 시작
for i in range(len(actual_flat)):              # 1200개 값을 하나씩 순서대로
    error_i = actual_flat[i] - predicted_flat[i]   # 이 값의 오차(실제-예측)
    sum_of_squared_error = sum_of_squared_error + error_i * error_i   # 오차의 제곱을 누적

In [ ]:
mean_squared_error = sum_of_squared_error / len(actual_flat)   # 누적한 제곱오차의 평균
rmse_value = mean_squared_error ** 0.5                           # 평균제곱오차의 제곱근 = RMSE

In [ ]:
sum_of_actual = 0.0                          # 실제값 합계를 누적할 변수
for i in range(len(actual_flat)):              # 1200개 값을 하나씩 순서대로
    sum_of_actual = sum_of_actual + actual_flat[i]   # 실제값을 누적

In [ ]:
average_actual = sum_of_actual / len(actual_flat)   # 실제 발전량의 평균값
nrmse_percent = 100.0 * rmse_value / average_actual   # RMSE를 평균으로 나누고 100을 곱해 %로 표현

=====================================================================
8. optimality gap 계산 (논문 Eq.1a 이익함수 3항 + Eq.13 오라클 {0,S})
=====================================================================

In [ ]:
da_flat = test_da_price.flatten()        # (100,12) DA가격 표를 1200개짜리 한 줄로 펼침
rt_flat = test_rt_price.flatten()        # (100,12) RT가격 표를 1200개짜리 한 줄로 펼침

In [ ]:
sum_of_realized_profit = 0.0             # 실제(AR 예측 기반) 총 이익을 누적할 변수
sum_of_oracle_profit = 0.0               # 오라클(사후 최적) 총 이익을 누적할 변수

In [ ]:
for i in range(len(actual_flat)):          # 테스트 1200개 관측치를 하나씩 순서대로 처리

    actual_i = actual_flat[i]                # 이 시간의 실제 발전량
    commitment_i = predicted_flat[i]         # 이 시간의 AR 예측값 = 일간전 약정량(commitment)
    da_i = da_flat[i]                        # 이 시간의 DA 가격
    rt_i = rt_flat[i]                        # 이 시간의 RT 가격
    penalty_cost_i = PENALTY_RATE * da_i     # 이 시간의 부족분 벌금단가 (DA가격의 50%)

    mismatch_i = actual_i - commitment_i       # 실제 - 약정 (양수면 잉여, 음수면 부족)
    surplus_i = max(mismatch_i, 0.0)             # 잉여량
    shortage_i = max(-mismatch_i, 0.0)           # 부족량
    realized_profit_i = CAPACITY_MW * DURATION_HOURS * (
        da_i * commitment_i + rt_i * surplus_i - penalty_cost_i * shortage_i
    )                                             # 논문 Eq.(1a) 3항 이익함수
    sum_of_realized_profit = sum_of_realized_profit + realized_profit_i

    profit_if_commit_zero = CAPACITY_MW * DURATION_HOURS * (rt_i * actual_i)     # 약정 0일 때 이익
    profit_if_commit_actual = CAPACITY_MW * DURATION_HOURS * (da_i * actual_i)    # 약정=실제발전량일 때 이익
    oracle_profit_i = max(profit_if_commit_zero, profit_if_commit_actual)          # 둘 중 더 이익나는 쪽
    sum_of_oracle_profit = sum_of_oracle_profit + oracle_profit_i

In [ ]:
optimality_gap_percent = 100.0 * (sum_of_oracle_profit - sum_of_realized_profit) / sum_of_oracle_profit

=====================================================================
9. 결과 출력 - bounded LAD(기본_모형_AR.py) 결과와 나란히 비교
=====================================================================

In [ ]:
print()
print("=== 기본 모형 AR (일반 LAD, 제약 없음) - 블록9 결과 ===")
print(f"nRMSE          = {nrmse_percent:.6f} %   (논문: {PAPER_NRMSE:.2f} %, bounded LAD: {BOUNDED_LAD_NRMSE:.6f} %)")
print(f"optimality gap = {optimality_gap_percent:.6f} %   (논문: {PAPER_GAP:.2f} %, bounded LAD: {BOUNDED_LAD_GAP:.6f} %)")
print()
print("| 회귀 방식 | nRMSE | optimality gap |")
print("|---|---:|---:|")
print(f"| 논문 (참고) | {PAPER_NRMSE:.2f}% | {PAPER_GAP:.2f}% |")
print(f"| bounded LAD (기본_모형_AR.py, 논문에 없는 [0,1] 제약 포함) | {BOUNDED_LAD_NRMSE:.6f}% | {BOUNDED_LAD_GAP:.6f}% |")
print(f"| **일반 LAD (제약 없음, 이 파일)** | **{nrmse_percent:.6f}%** | **{optimality_gap_percent:.6f}%** |")